# DERMDOC dubbing on Colab (T4): 2 videos -> Bengali, Tamil, Telugu, Malayalam, Odia
**Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**. Nothing to upload or edit: the code, the dub
plans and the source-video URLs are embedded. Each step prints its progress; a failure in one video never stops the
others. At the end `/kaggle/working/dub_out.zip` (the dubbed mp4s + QA reports) downloads automatically.

Videos: e7e42eaf, f7ce58d4

Expected on a free T4 (estimate): ~3-5 min of installs, then roughly 1 min per video. If the session drops, just
**Run all** again: finished videos are skipped.

In [ ]:
# 1/6  GPU check
import os, subprocess, sys, time
from pathlib import Path

IDS = ["e7e42eaf", "f7ce58d4"]
ROOT = Path("/tmp/dub")
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONUNBUFFERED"] = "1"


def run(cmd, cwd=ROOT):
    """Run a command and stream its output. Never raises: one failed step must not stop Run all."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    try:
        p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True)
        rc = p.wait()
    except Exception as e:
        print(f"!! {e!r}", flush=True)
        rc = -1
    print(f"[exit {rc} after {time.time() - t:.0f}s]", flush=True)
    return rc


print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
cc = torch.cuda.get_device_capability(0)
print(f"python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | "
      f"{torch.cuda.get_device_name(0)} cc {cc[0]}.{cc[1]} | "
      f"{'bf16 ok' if cc >= (8, 0) else 'no bf16 -> Indic-Mio fp16 (fp32 fallback), MioCodec fp32'} | "
      f"{torch.cuda.mem_get_info()[1] / 2**30:.1f} GB GPU memory")
print(f"{len(IDS)} videos: {', '.join(IDS)}")

In [ ]:
# 2/6  Install (Colab's own torch / torchaudio are kept: pinned through a constraints file)
import glob, shutil, site
cons = [f"torch=={torch.__version__}"]
try:
    import torchaudio
    cons.append(f"torchaudio=={torchaudio.__version__}")
except ImportError:
    print("torchaudio not preinstalled: pip will pick the one matching torch")
Path("/tmp/constraints.txt").write_text("\n".join(cons) + "\n")
if shutil.which("ffmpeg") is None:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt", "faster-whisper==1.2.1",
     "demucs==4.1.0", "librosa", "soundfile", "pyloudnorm", "threadpoolctl", "packaging"])
# MioCodec declares Python >= 3.12; on an older Colab Python try it anyway
extra = [] if sys.version_info >= (3, 12) else ["--ignore-requires-python"]
run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt", *extra,
     "git+https://github.com/Aratako/MioCodec"])
# CTranslate2 (faster-whisper) needs cuBLAS + cuDNN 9: point it at the pip CUDA libraries that came with torch
libs = sorted({d for sp in site.getsitepackages() for d in glob.glob(f"{sp}/nvidia/*/lib")})
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")]).rstrip(":")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run([sys.executable, "-c", "import faster_whisper, demucs.api, miocodec, librosa, pyloudnorm, transformers, torch; "
     "print('imports ok | transformers', transformers.__version__, '| torch', torch.__version__)"])

In [ ]:
# 3/6  Write the engine code and every video's plan.json + meta.json (embedded below), then validate the plans
import base64, gzip, json
PAYLOAD = """
...
"""
files = json.loads(gzip.decompress(base64.b64decode(PAYLOAD)).decode("utf-8"))
for rel, text in files.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
print(f"wrote {len(files)} files under {ROOT}")
for vid in IDS:
    (ROOT / "work" / vid / "dub.lock").unlink(missing_ok=True)  # single engine here: clear claims of a dropped session
    r = subprocess.run([sys.executable, "validate_plan.py", str(ROOT / "work" / vid / "plan.json")], cwd=ROOT,
                       capture_output=True, text=True)
    msg = (r.stdout + r.stderr).strip().splitlines()
    print(f"{vid}: plan {'OK' if r.returncode == 0 else 'INVALID (video will be skipped)'}  {msg[0] if msg else ''}")

In [ ]:
# 4/6  Download the source videos, decode their audio (44.1 kHz stereo) and split vocals / background with
#       Demucs htdemucs --two-stems=vocals on the GPU -> work/<id>/stems/{vocals,no_vocals}.wav
run([sys.executable, "vm/prep_cuda.py", "work", "--gens", "colab_gens.json", "--only", ",".join(IDS), "--stems-only"])
for vid in IDS:
    ok = all((ROOT / "work" / vid / "stems" / f).exists() for f in ("vocals.wav", "no_vocals.wav"))
    print(f"{vid}: {'stems ready' if ok else 'NO STEMS (download or separation failed; video will be skipped)'}")

In [ ]:
# 5/6  Dub: Indic-Mio on transformers (fp16 on T4, fp32 fallback) -> MioCodec voice cloning per character ->
#       Whisper QA with retries -> pitch/fx -> mix. One video at a time (--group 1) so finished videos land early;
#       then a repair pass re-voices lines the QA flagged.
t0 = time.time()
run([sys.executable, "vm/dubplan_cuda.py", "work", "out", "--backend", "transformers", "--group", "1",
     "--only", ",".join(IDS)])
run([sys.executable, "vm/repair_cuda.py", "work", "out", "0.6", "--backend", "transformers", "--only", ",".join(IDS)])
LANGS = ["bn", "ta", "te", "ml", "or"]
done = {vid: [l for l in LANGS if (ROOT / "out" / f"{vid}_{l}.mp4").exists()] for vid in IDS}
for vid, langs in done.items():
    print(f"{vid}: {len(langs)}/5 languages {' '.join(langs)}")
print(f"dub + repair took {(time.time() - t0) / 60:.1f} min; {sum(map(len, done.values()))} mp4s")

In [ ]:
# 6/6  Zip out/<id>_<lang>.mp4 + out/<id>_<lang>/qa.json and download it
import zipfile
ZIP = Path("/kaggle/working/dub_out.zip")
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for vid in IDS:
        for lang in ["bn", "ta", "te", "ml", "or"]:
            mp4, qa = ROOT / "out" / f"{vid}_{lang}.mp4", ROOT / "out" / f"{vid}_{lang}" / "qa.json"
            if mp4.exists():
                zf.write(mp4, f"out/{mp4.name}", compress_type=zipfile.ZIP_STORED)
                n += 1
            if qa.exists():
                zf.write(qa, f"out/{vid}_{lang}/qa.json")
print(f"{ZIP}: {n} videos, {ZIP.stat().st_size / 2**20:.1f} MB")
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print(f"automatic download unavailable ({e!r}); use the Files panel on the left: /kaggle/working/dub_out.zip")

In [ ]:
# Optional: also copy the zip to Google Drive. Off by default (mounting asks for authorisation, which would block
# Run all): set COPY_TO_DRIVE = True and run this cell by hand.
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copy(ZIP, "/content/drive/MyDrive/dub_out_colab.zip")
        print("copied to MyDrive/dub_out_colab.zip")
    except Exception as e:
        print(f"Drive copy failed: {e!r}")
else:
    print("Drive copy skipped (set COPY_TO_DRIVE = True to enable)")